Cleaning Data
import libraries and datasets

In [2]:
import pandas as pd
import numpy as np

lfs = pd.read_parquet('../data/processed/lfs_combined.parquet')
cpi = pd.read_parquet('../data/processed/cpi_clean.parquet')
wages = pd.read_parquet('../data/processed/wages_clean.parquet')

print(f"LFS: {len(lfs):,} rows, {lfs.shape[1]} columns")
print(f"CPI: {len(cpi)} rows")
print(f"Wages: {len(wages)} rows")
print("\nLFS dtypes:")
print(lfs.dtypes)

LFS: 6,759,638 rows, 17 columns
CPI: 192 rows
Wages: 1235 rows

LFS dtypes:
LFSSTAT               int8
PROV                  int8
AGE_12                int8
MARSTAT               int8
EDUC                  int8
COWMAIN            float32
IMMIG                 int8
NAICS_21           float32
NOC_10             float32
UHRSMAIN           float32
FTPTMAIN           float32
TENURE             float32
HRLYEARN           float32
PERMTEMP           float32
year                 int16
month                 int8
date        datetime64[ns]
dtype: object


## Step 2 — Assess Data Quality

Before cleaning anything we need to understand what's missing.
In real government microdata, missing values are often intentional —
they represent "not applicable" rather than data errors.
We need to distinguish between the two using the codebook.

In [3]:
# Check missing values across all columns
missing = lfs.isnull().sum()
missing_pct = (missing / len(lfs) * 100).round(2)

summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

print("Missing values per column:")
print(summary[summary['missing_count'] > 0].sort_values('missing_pct', ascending=False))

Missing values per column:
          missing_count  missing_pct
HRLYEARN        3353955        49.62
PERMTEMP        3353955        49.62
UHRSMAIN        2812952        41.61
TENURE          2812952        41.61
FTPTMAIN        2812952        41.61
NOC_10          2363870        34.97
COWMAIN         2363870        34.97
NAICS_21        2363870        34.97


## Step 3 — Decode Categorical Columns

The LFS stores all categorical variables as numeric codes.
For example PROV=35 means Ontario, LFSSTAT=1 means Employed.
We check the unique values in each column so we can map them
to human-readable labels using the codebook.

In [4]:
cat_cols = ['LFSSTAT', 'PROV', 'AGE_12', 'MARSTAT', 
            'EDUC', 'IMMIG', 'FTPTMAIN', 'PERMTEMP']

for col in cat_cols:
    if col in lfs.columns:
        print(f"\n{col}: {sorted(lfs[col].dropna().unique().tolist())}")


LFSSTAT: [1, 2, 3, 4]

PROV: [10, 11, 12, 13, 24, 35, 46, 47, 48, 59]

AGE_12: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

MARSTAT: [1, 2, 3, 4, 5, 6]

EDUC: [0, 1, 2, 3, 4, 5, 6]

IMMIG: [1, 2, 3]

FTPTMAIN: [1.0, 2.0]

PERMTEMP: [1.0, 2.0, 3.0, 4.0]


## Step 4 — Apply Codebook Labels

Mapping all numeric codes to human-readable labels
based on the LFS PUMF codebook documentation.
This makes the data interpretable for analysis and visualisation.

In [5]:
# LFSSTAT — Labour Force Status (the most important variable)
lfsstat_map = {
    1: 'Employed',
    2: 'Unemployed',
    3: 'Not in labour force',
    4: 'Not in labour force'
}

# PROV — Province
prov_map = {
    10: 'Newfoundland',
    11: 'PEI',
    12: 'Nova Scotia',
    13: 'New Brunswick',
    24: 'Quebec',
    35: 'Ontario',
    46: 'Manitoba',
    47: 'Saskatchewan',
    48: 'Alberta',
    59: 'British Columbia'
}

# AGE_12 — Age groups
age_map = {
    1:  '15-19',
    2:  '20-24',
    3:  '25-29',
    4:  '30-34',
    5:  '35-39',
    6:  '40-44',
    7:  '45-49',
    8:  '50-54',
    9:  '55-59',
    10: '60-64',
    11: '65-69',
    12: '70+'
}

# MARSTAT — Marital Status
marstat_map = {
    1: 'Married',
    2: 'Common-law',
    3: 'Widowed',
    4: 'Separated',
    5: 'Divorced',
    6: 'Single'
}

# EDUC — Education Level
educ_map = {
    0: 'Less than high school',
    1: 'High school diploma',
    2: 'Some post-secondary',
    3: 'Trade certificate',
    4: 'College diploma',
    5: 'University below bachelor',
    6: 'Bachelor degree or higher'
}

# IMMIG — Immigration Status
immig_map = {
    1: 'Canadian-born',
    2: 'Immigrant',
    3: 'Non-permanent resident'
}

# FTPTMAIN — Full time vs Part time
ftpt_map = {
    1.0: 'Full-time',
    2.0: 'Part-time'
}

# PERMTEMP — Permanent vs Temporary job
permtemp_map = {
    1.0: 'Permanent',
    2.0: 'Seasonal',
    3.0: 'Temporary',
    4.0: 'Other temporary'
}

print("All mappings defined successfully")

All mappings defined successfully


## Step 5 — Apply Labels to DataFrame

Creating new human-readable columns alongside the original
numeric codes. We keep both so we can use numeric codes
for modelling and labels for visualisation.

In [6]:
lfs['lfsstat_label'] = lfs['LFSSTAT'].map(lfsstat_map)
lfs['prov_label'] = lfs['PROV'].map(prov_map)
lfs['age_label'] = lfs['AGE_12'].map(age_map)
lfs['marstat_label'] = lfs['MARSTAT'].map(marstat_map)
lfs['educ_label'] = lfs['EDUC'].map(educ_map)
lfs['immig_label'] = lfs['IMMIG'].map(immig_map)
lfs['ftpt_label'] = lfs['FTPTMAIN'].map(ftpt_map)
lfs['permtemp_label'] = lfs['PERMTEMP'].map(permtemp_map)

# Verify mappings worked
print("Sample of decoded data:")
print(lfs[['LFSSTAT', 'lfsstat_label', 'PROV', 'prov_label', 
           'IMMIG', 'immig_label', 'EDUC', 'educ_label']].head(10))

Sample of decoded data:
   LFSSTAT        lfsstat_label  PROV        prov_label  IMMIG  \
0        4  Not in labour force    24            Quebec      3   
1        3  Not in labour force    48           Alberta      3   
2        4  Not in labour force    35           Ontario      2   
3        4  Not in labour force    35           Ontario      3   
4        4  Not in labour force    35           Ontario      3   
5        1             Employed    13     New Brunswick      3   
6        4  Not in labour force    12       Nova Scotia      3   
7        4  Not in labour force    11               PEI      3   
8        1             Employed    59  British Columbia      3   
9        4  Not in labour force    12       Nova Scotia      3   

              immig_label  EDUC                 educ_label  
0  Non-permanent resident     5  University below bachelor  
1  Non-permanent resident     3          Trade certificate  
2               Immigrant     2        Some post-secondary  
3  No

## Step 6 — Create Key Analysis Flags

Creating binary flags and derived variables that we'll
use directly in our analysis questions.

In [7]:
# Binary employed flag — easier to work with than 4-way LFSSTAT
lfs['is_employed'] = (lfs['LFSSTAT'] == 1).astype(int)
lfs['is_unemployed'] = (lfs['LFSSTAT'] == 2).astype(int)

# Working age population flag (25-54 is prime working age)
lfs['is_prime_age'] = lfs['AGE_12'].between(4, 9).astype(int)

# Immigrant flag
lfs['is_immigrant'] = (lfs['IMMIG'] == 2).astype(int)

# Higher education flag
lfs['has_degree'] = (lfs['EDUC'] >= 6).astype(int)

# Year-month string for plotting
lfs['ym'] = lfs['date'].dt.to_period('M').astype(str)

print("New columns created:")
print(lfs[['is_employed', 'is_unemployed', 'is_prime_age', 
           'is_immigrant', 'has_degree']].sum())
print(f"\nTotal employed records: {lfs['is_employed'].sum():,}")
print(f"Total unemployed records: {lfs['is_unemployed'].sum():,}")

New columns created:
is_employed      3579637
is_unemployed     367049
is_prime_age     3104702
is_immigrant     1030329
has_degree        601344
dtype: int64

Total employed records: 3,579,637
Total unemployed records: 367,049


## Step 7 — Save Cleaned Dataset

Saving the fully decoded and cleaned LFS dataset to the
processed folder. This is what all subsequent notebooks
will load for analysis and visualisation.

In [8]:
lfs.to_parquet('../data/processed/lfs_clean.parquet', index=False)

print("Cleaned dataset saved!")
print(f"\nFinal shape: {lfs.shape}")
print(f"\nAll columns:")
for col in lfs.columns:
    print(f"  {col}")

Cleaned dataset saved!

Final shape: (6759638, 31)

All columns:
  LFSSTAT
  PROV
  AGE_12
  MARSTAT
  EDUC
  COWMAIN
  IMMIG
  NAICS_21
  NOC_10
  UHRSMAIN
  FTPTMAIN
  TENURE
  HRLYEARN
  PERMTEMP
  year
  month
  date
  lfsstat_label
  prov_label
  age_label
  marstat_label
  educ_label
  immig_label
  ftpt_label
  permtemp_label
  is_employed
  is_unemployed
  is_prime_age
  is_immigrant
  has_degree
  ym


## Step 8 — Outlier Check on Continuous Variables

Checking hourly earnings and hours worked for extreme values
that could skew our analysis. Real survey data often contains
top-coded values where Statistics Canada caps extreme responses
to protect respondent anonymity.

In [9]:
# Check distribution of continuous variables
continuous = ['HRLYEARN', 'UHRSMAIN']

for col in continuous:
    data = lfs[col].dropna()
    print(f"\n{col}:")
    print(f"  Min:    {data.min():.2f}")
    print(f"  Max:    {data.max():.2f}")
    print(f"  Mean:   {data.mean():.2f}")
    print(f"  Median: {data.median():.2f}")
    print(f"  Std:    {data.std():.2f}")
    print(f"  Values over 100: {(data > 100).sum():,}")


HRLYEARN:
  Min:    321.00
  Max:    27137.00
  Mean:   3324.61
  Median: 2850.00
  Std:    1787.61
  Values over 100: 3,405,683

UHRSMAIN:
  Min:    1.00
  Max:    990.00
  Mean:   358.00
  Median: 400.00
  Std:    116.10
  Values over 100: 3,758,262


## Step 9 — Temporal Coverage Check

Verifying we have consistent record counts across all months.
Sudden drops in a particular month could indicate data quality
issues or survey methodology changes we need to account for.

In [10]:
# Records per month — check for consistency
monthly_counts = lfs.groupby(['year', 'month']).size().reset_index(name='record_count')

print("Records per month:")
print(monthly_counts.to_string())

# Flag any months significantly below average
avg = monthly_counts['record_count'].mean()
low_months = monthly_counts[monthly_counts['record_count'] < avg * 0.85]

if len(low_months) > 0:
    print(f"\nWarning — months significantly below average ({avg:.0f}):")
    print(low_months)
else:
    print(f"\nAll months consistent. Average records per month: {avg:,.0f}")

Records per month:
    year  month  record_count
0   2021      1         83787
1   2021      2         85326
2   2021      3         86552
3   2021      4         87597
4   2021      5         88057
5   2021      6         87573
6   2021      7         86405
7   2021      8         86862
8   2021      9         86049
9   2021     10         86555
10  2021     11         89546
11  2021     12         92685
12  2022      1         97899
13  2022      2        103658
14  2022      3        107967
15  2022      4        111708
16  2022      5        111139
17  2022      6        111740
18  2022      7        111294
19  2022      8        111173
20  2022      9        110439
21  2022     10        110059
22  2022     11        109348
23  2022     12        110147
24  2023      1        108064
25  2023      2        104851
26  2023      3        101043
27  2023      4        100024
28  2023      5         98785
29  2023      6         97522
30  2023      7         99953
31  2023      8      

## Step 10 — Employment Rate Baseline

Calculating the overall employment rate for each month
as a baseline reference. This is the headline number we'll
compare all our sub-group analyses against in Notebook 3.

In [11]:
# Overall monthly employment rate
# Only count people who are employed or unemployed (in labour force)
in_lf = lfs[lfs['LFSSTAT'].isin([1, 2])]

monthly_emp = in_lf.groupby('date').agg(
    employed=('is_employed', 'sum'),
    total_lf=('is_employed', 'count')
).reset_index()

monthly_emp['employment_rate'] = (monthly_emp['employed'] / monthly_emp['total_lf'] * 100).round(2)

print("Monthly employment rate (% of labour force employed):")
print(monthly_emp[['date', 'employed', 'total_lf', 'employment_rate']].to_string())

Monthly employment rate (% of labour force employed):
         date  employed  total_lf  employment_rate
0  2021-01-01     41400     45696            90.60
1  2021-02-01     42969     47245            90.95
2  2021-03-01     44334     48382            91.63
3  2021-04-01     44516     49019            90.81
4  2021-05-01     45998     50080            91.85
5  2021-06-01     46642     50815            91.79
6  2021-07-01     42694     49771            85.78
7  2021-08-01     42591     50155            84.92
8  2021-09-01     45805     49919            91.76
9  2021-10-01     46192     50221            91.98
10 2021-11-01     47887     51944            92.19
11 2021-12-01     49515     53541            92.48
12 2022-01-01     49474     55267            89.52
13 2022-02-01     55259     60057            92.01
14 2022-03-01     56346     62651            89.94
15 2022-04-01     59563     65421            91.05
16 2022-05-01     61090     66512            91.85
17 2022-06-01     62171     

## Step 11 — Fix Variable Scales

HRLYEARN is stored in cents in the raw LFS data.
Converting to dollars for interpretability.
UHRSMAIN values also require scale verification against codebook.

In [12]:
# Convert hourly earnings from cents to dollars
lfs['hourly_wage'] = lfs['HRLYEARN'] / 100

# Verify
print("Hourly wage after conversion:")
print(f"  Min:    ${lfs['hourly_wage'].dropna().min():.2f}")
print(f"  Max:    ${lfs['hourly_wage'].dropna().max():.2f}")
print(f"  Mean:   ${lfs['hourly_wage'].dropna().mean():.2f}")
print(f"  Median: ${lfs['hourly_wage'].dropna().median():.2f}")

# Check UHRSMAIN distribution more carefully
print("\nUHRSMAIN value counts (top 20):")
print(lfs['UHRSMAIN'].value_counts().sort_index().head(20))

Hourly wage after conversion:
  Min:    $3.21
  Max:    $271.37
  Mean:   $33.25
  Median: $28.50

UHRSMAIN value counts (top 20):
UHRSMAIN
1.0       92
2.0       25
3.0       24
4.0        3
5.0      379
6.0        5
7.0       23
8.0        9
9.0        2
10.0    4253
11.0       3
12.0      32
13.0      13
14.0       4
15.0     604
16.0       6
17.0       7
18.0       3
19.0       2
20.0    7341
Name: count, dtype: int64


## Step 12 — Clean Hours Worked Variable

UHRSMAIN values above 100 are special non-response codes
per the LFS codebook. Setting these to NaN so they don't
skew analysis of actual working hours.

In [13]:
# Values above 100 are special codes, not real hours
lfs['weekly_hours'] = lfs['UHRSMAIN'].copy()
lfs.loc[lfs['weekly_hours'] > 100, 'weekly_hours'] = np.nan

print("Weekly hours after cleaning:")
print(f"  Min:    {lfs['weekly_hours'].dropna().min():.0f} hours")
print(f"  Max:    {lfs['weekly_hours'].dropna().max():.0f} hours")
print(f"  Mean:   {lfs['weekly_hours'].dropna().mean():.1f} hours")
print(f"  Median: {lfs['weekly_hours'].dropna().median():.0f} hours")
print(f"  Valid records: {lfs['weekly_hours'].notna().sum():,}")

Weekly hours after cleaning:
  Min:    1 hours
  Max:    100 hours
  Mean:   70.5 hours
  Median: 80 hours
  Valid records: 188,424


## Step 13 — Save Final Cleaned Dataset

All cleaning steps complete. Saving the fully processed
dataset ready for analysis in Notebook 3.

Cleaning steps applied:
- Decoded all categorical columns using LFS codebook
- Created binary analysis flags (employed, immigrant, degree etc.)
- Converted hourly earnings from cents to dollars
- Removed invalid codes from hours worked variable
- Verified temporal consistency across all 64 months

In [14]:
lfs.to_parquet('../data/processed/lfs_clean.parquet', index=False)

print("Final cleaned dataset saved!")
print(f"\nShape: {lfs.shape}")
print(f"\nColumns available for analysis:")
for col in sorted(lfs.columns):
    print(f"  {col}")

Final cleaned dataset saved!

Shape: (6759638, 33)

Columns available for analysis:
  AGE_12
  COWMAIN
  EDUC
  FTPTMAIN
  HRLYEARN
  IMMIG
  LFSSTAT
  MARSTAT
  NAICS_21
  NOC_10
  PERMTEMP
  PROV
  TENURE
  UHRSMAIN
  age_label
  date
  educ_label
  ftpt_label
  has_degree
  hourly_wage
  immig_label
  is_employed
  is_immigrant
  is_prime_age
  is_unemployed
  lfsstat_label
  marstat_label
  month
  permtemp_label
  prov_label
  weekly_hours
  year
  ym
